# 05 — Estudo de caso: Rio Claro

**Objetivo:** aprofundar o retrato de Rio Claro dentro do panorama estadual —
o "Nível 2" do estudo (o "Nível 1" é o notebook 04, com os 645 municípios).

**Entrada:** `data/processed/dataset_consolidado_sp.csv`

**Estrutura desta seção no artigo:** primeiro mostra a associação em escala
estadual (notebook 04), depois usa Rio Claro para dar profundidade — comparar
com a média/mediana do estado e projetar a tendência.

> 📌 Quando o CadÚnico de Rio Claro chegar, ele entra **aqui** — como uma
> seção nova neste notebook, cruzando renda/composição domiciliar individual
> com o que já temos. Até lá, esta análise usa apenas dados públicos
> agregados (IBGE + DATASUS).


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv")
rio_claro = df[df["codigo_ibge"] == config.RIO_CLARO_CODIGO_IBGE].sort_values("ano")
rio_claro


## 5.1 Rio Claro × média e mediana do estado


In [ ]:
resumo_estado = df.groupby("ano").agg(
    media_estado=("taxa_internacao_100k_idosos", "mean"),
    mediana_estado=("taxa_internacao_100k_idosos", "median"),
).reset_index()

comparativo = resumo_estado.merge(
    rio_claro[["ano", "taxa_internacao_100k_idosos"]].rename(columns={"taxa_internacao_100k_idosos": "rio_claro"}),
    on="ano", how="left",
)
comparativo


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(comparativo["ano"], comparativo["media_estado"], marker="o", label="Média do estado (SP)")
ax.plot(comparativo["ano"], comparativo["rio_claro"], marker="o", label=config.RIO_CLARO_NOME, color="red")
ax.set_xlabel("Ano")
ax.set_ylabel("Internações por 100 mil idosos (causas evitáveis)")
ax.set_title("Rio Claro × média do estado de SP")
ax.legend()
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "rio_claro_vs_estado.png", dpi=150)
plt.show()


## 5.2 Projeção simples até 2026

Regressão linear simples sobre a série histórica de Rio Claro (2019-2022)
para estimar a tendência — **isso é só uma referência ilustrativa**, não uma
previsão robusta (a série tem poucos pontos). Vale declarar essa limitação
no artigo.


In [ ]:
from numpy.polynomial import polynomial as P

serie = rio_claro.dropna(subset=["taxa_internacao_100k_idosos"])
if len(serie) >= 2:
    coefs = np.polyfit(serie["ano"], serie["taxa_internacao_100k_idosos"], 1)
    anos_futuros = np.arange(serie["ano"].min(), 2027)
    projecao = np.polyval(coefs, anos_futuros)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(serie["ano"], serie["taxa_internacao_100k_idosos"], label="Observado")
    ax.plot(anos_futuros, projecao, "--", color="red", label="Tendência linear (projeção)")
    ax.set_xlabel("Ano")
    ax.set_ylabel("Internações por 100 mil idosos")
    ax.set_title(f"Projeção de tendência — {config.RIO_CLARO_NOME} (ilustrativa)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(config.OUTPUTS_FIGURES / "projecao_rio_claro.png", dpi=150)
    plt.show()
else:
    print("Poucos pontos para projeção — confirme se todos os anos de 2019-2022 foram coletados no notebook 02.")


## 5.3 Perfil das internações em Rio Claro, por causa


In [ ]:
causas = [c for c in config.CAUSAS_CID10.keys() if c in rio_claro.columns]
if causas:
    perfil = rio_claro[causas].sum().rename(index=config.CAUSAS_LABELS)
    fig, ax = plt.subplots(figsize=(7, 5))
    perfil.sort_values().plot(kind="barh", ax=ax, color="#C44E52")
    ax.set_xlabel("Total de internações (2019-2022)")
    ax.set_title(f"Perfil das internações evitáveis em idosos — {config.RIO_CLARO_NOME}")
    fig.tight_layout()
    fig.savefig(config.OUTPUTS_FIGURES / "perfil_causas_rio_claro.png", dpi=150)
    plt.show()


## 5.4 Espaço reservado — CadÚnico (quando disponível)

Quando os dados do CadÚnico de Rio Claro chegarem, adicione aqui:
- leitura do arquivo (`data/external/cadunico_rio_claro.*`)
- checagem de que é agregado/anonimizado (ver observações de ética já
  discutidas) antes de qualquer cruzamento
- cruzamento com o perfil por bairro/setor censitário já obtido acima
